# 03 — Results figures

Read the persisted experiment artefacts: prediction metrics per split,
retrieval/agent ablations, and the predicted-vs-true scatter grid.

Regenerate everything with:

```bash
python scripts/run_ablations.py --run-all --full-agent
python scripts/plot_pred_vs_true.py
```

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("project root:", ROOT)


## Prediction metrics (latest row per model/split)

In [ ]:
import pandas as pd

metrics = pd.read_csv(ROOT / "results/metrics.csv")
latest = (metrics.sort_values("timestamp")
                 .groupby(["model", "split"], as_index=False)
                 .tail(1))
latest[["model", "split", "pearson", "spearman", "mse", "mae",
        "top50_precision", "top50_recall"]].sort_values(["split", "model"])

In [ ]:
import matplotlib.pyplot as plt

pivot = latest.pivot(index="split", columns="model", values="pearson")
pivot.plot.bar(rot=0, ylabel="Pearson r (pseudobulk delta)", title="Prediction quality by split")
plt.tight_layout()
out = ROOT / "results" / "figures" / "pearson_by_split.png"
out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=150)
print("saved", out)

## Ablation levels

`ablation_results.json` carries all four levels: prediction (baseline, FiLM
head), retrieval (FAISS vs random vs expression correlation) and agent
(mock probes and the real tool-calling agent).

In [ ]:
import json

path = ROOT / "results/ablation_results.json"
if path.exists():
    ablation = json.loads(path.read_text())
    print("prediction models:", sorted(ablation.get("prediction", {})))
    print("retrieval systems:", sorted(ablation.get("retrieval", {})))
    print("agent groups:", sorted(ablation.get("agent", {})))
else:
    print("run scripts/run_ablations.py first")

In [ ]:
if path.exists():
    for system, per_k in ablation.get("retrieval", {}).items():
        recall5 = per_k.get("5", {}).get("recall")
        mrr5 = per_k.get("5", {}).get("mrr")
        print(f"{system:<20s} recall@5={recall5}  mrr@5={mrr5}")
    print()
    for group, values in ablation.get("agent", {}).items():
        print(group, {k: round(v, 3) for k, v in values.items()})

## Predicted vs. true deltas

Each panel is one test-split perturbation; predictions are averaged over
control cells and compared gene-by-gene against the observed pseudobulk
delta.

In [ ]:
from IPython.display import Image, display

png = ROOT / "results/pred_vs_true.png"
if png.exists():
    display(Image(filename=str(png)))
else:
    print("run scripts/plot_pred_vs_true.py first")